## Stop-Level Feature Engineering for Transit Accessibility Modeling in Nairobi

**Objective**

The goal of this notebook is to build a complete, feature-rich dataset describing both existing matatu stops and potential (candidate) stop locations across Nairobi.  
These features integrate road network structure, GTFS service intensity, population and ward context, traffic flows, and spatial accessibility.  
The final output is a training dataset suitable for machine learning-especially **graph neural networks (GNNs)** or geospatial predictive models; focused on stop suitability and transit coverage optimization.

**Why Stop-Focused Aggregation?**

Transit systems are inherently node-based. Stops act as the primary “interaction points” between people and the transport network.  
Aggregating features at the stop level:

- Captures local context where passengers board/alight  
- Preserves network structure (roads → stops → routes → mobility patterns)  
- Allows natural integration of spatial, demographic, and service layers  
- Enables learning models (GNNs, spatial ML) to reason over the transit topology

**Graph Neural Networks benefit especially** because:

- Stops become **nodes** with rich features  
- Road links, route shapes, or stop adjacencies become **edges**  
- GNNs exploit these structured connections to understand accessibility, coverage, and demand patterns better than flat tabular models

This notebook prepares the foundation needed for such modeling.

### i. Libraries

Imports and Constants

Load all libraries needed for data processing, spatial analysis, routing, and parallelization.  
Also define two global constants used throughout the notebook:

- `CBD_COORDS`: Nairobi CBD reference point  
- `PEAK_HOURS`: Morning and evening rush-hour windows

In [ ]:
import pandas as pd       # Data manipulation and analysis
import numpy as np        # Numerical arrays and fast math operations
from scipy.spatial import cKDTree        # Fast nearest-neighbor search for spatial queries
from scipy.spatial.distance import cdist # Computes pairwise distances efficiently
import osmnx as ox        # OpenStreetMap graph loading and routing utilities
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner notebook output
import gtfs_kit as gk     # GTFS feed loading, filtering, and timetable analysis
import polars as pl       # High-performance DataFrame engine (faster than pandas)
import geopandas as gpd   # Geospatial DataFrame support (points, polygons, CRS)
from multiprocessing import Pool, cpu_count  # Parallel processing for heavy spatial ops
from functools import partial                # Helps pass extra parameters into parallelized functions
from tqdm import tqdm     # Progress bars for loops and parallel tasks
pd.set_option('display.max_columns', None)  # Make sure full DataFrames display in the notebook
# Constants and configuration for Nairobi transport analysis
CBD_COORDS = np.array([[-1.2864, 36.8172]])  
# Coordinates of Nairobi CBD (latitude, longitude) — main accessibility reference point.
PEAK_HOURS = [6, 9, 15]
# Typical morning + evening rush hours used for filtering GTFS trips or traffic models.


### ii. Load GTFS Data

This cell loads the GTFS feed (Digital Matatus 2019), extracts the main tables
(routes, trips, stops, stop times, shapes), and prints a quick summary to
confirm everything loaded correctly.


In [2]:
# load data
print("Loading data...")
# load GTFS feed ---
print("Loading GTFS data...")
feed_path = '/home/dataopske/Desktop/jav/data/raw/digitalmatatu/GTFS_FEED_2019.zip'
feed = gk.read_feed(feed_path, dist_units='km')  # Read GTFS feed with distances in km

# Extract GTFS components
gtfs_routes = feed.routes        # Route definitions
gtfs_trips = feed.trips          # Trip schedules
gtfs_stop_times = feed.stop_times # Stop-level arrival/departure times
gtfs_stops = feed.stops          # Stop coordinates + metadata
gtfs_shapes = feed.shapes        # Route shape geometry (polyline)

# Quick summary
print(f"✓ GTFS loaded: {len(gtfs_routes)} routes, {len(gtfs_stops)} stops")
print(f" Route IDs: {gtfs_routes['route_id'].unique()[:5]}...")

Loading data...
Loading GTFS data...
✓ GTFS loaded: 136 routes, 4284 stops
 Route IDs: <StringArray>
['10000107D11', '10000114011', '10000116011', '10100011A11', '10200010811']
Length: 5, dtype: string...


> The Digital Matatus GTFS feed was successfully loaded, containing 136 routes and 4,284 stops. The preview shows the first few route IDs to confirm integrity of the feed.

### iii. Ward Merge, Processed Data, and Road Network

Here we merge ward information into the GTFS stops so every stop is tied to an administrative area.  
We also load the preprocessed traffic and spatio-temporal datasets needed for later modeling.  
Finally, we load (or download if missing) the Nairobi road network for routing and distance calculations.


In [3]:
# load ward lookup early for gtfs_stops
ward_lookup = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/stop_ward_lookup.csv')
gtfs_stops = gtfs_stops.merge(ward_lookup[['stop_id', 'ward']], on='stop_id', how='left')
gtfs_stops['ward'] = gtfs_stops['ward'].fillna('unknown')
print(f"✓ Wards merged to GTFS stops: {gtfs_stops['ward'].nunique()} unique wards")

# load the daily aggregated one:
model2_daily = pd.read_parquet('/home/dataopske/Desktop/jav/data/processed/model2_traffic_daily.parquet')
df1 = gpd.read_parquet('/home/dataopske/Desktop/jav/data/processed/spatio_temporal_calendar.parquet')
print("✓ Processed data (traffic, spatio_temporal) loaded.")

# load osm road network
print("Downloading road network...")
try:
    G = ox.load_graphml('/home/dataopske/Desktop/jav/data/processed/nairobi_drive.graphml')
except:
    G = ox.graph_from_place("Nairobi, Kenya", network_type='drive')
    ox.save_graphml(G, 'nairobi_roads.graphml')
    
nodes_gdf = ox.graph_to_gdfs(G, edges=False) # Convert nodes to GeoDataFrame
print("✓ Road network loaded.")

✓ Wards merged to GTFS stops: 79 unique wards
✓ Processed data (traffic, spatio_temporal) loaded.
✓ Road network loaded.


> The ward lookup successfully merged into the GTFS stops, giving us 79 distinct wards.  
The processed traffic and spatio-temporal datasets loaded correctly and are ready for use.  
The notebook then retrieved the Nairobi road network—either from disk or by downloading it—confirming that the routing layer is now fully available.


### iv. Helper Functions for Feature Extraction

Here we define all the helper functions used to compute spatial, service, traffic, population, and ward-level features for each stop or candidate location.  
This includes haversine distance, road-network context, stop spacing, neighborhood population, GTFS service intensity, and traffic lookup.  
We also add composite derived metrics and the main function that bundles all features for a stop.  
Finally, we include a generator for negative (candidate) points within each ward.

**1. Distance & Spatial Basics**
- `haversine_km`: Computes the great-circle distance between two coordinates in kilometers.
- `get_spatial_features`: Returns simple spatial metrics such as distance to the Nairobi CBD.

**2. Road Network Context**
- `get_road_features`: Uses OSMnx to find the nearest road node, its degree (intersection or not), the road type (e.g., primary, secondary), and distance to the nearest major road.  
  This captures how well-connected and road-accessible a location is.

**3. Stop Density & Local Spacing**
- `get_stop_spacing_features`: Uses a KD-tree of all stops to compute distances to the nearest existing stops and spacing regularity.
- `get_stops_in_radius`: Counts how many stops exist within a given radius (e.g., 500m, 1km).

**4. Population & Ward-Level Context**
- `get_population_features`: Uses aggregated ward data to approximate how many people live within 500m/1km, local poverty rate weighting, and how many people are underserved.
- `get_ward_features`: Loads broader ward-level attributes such as access scores, population, poverty, and categories.

**5. Transit Service Features**
- `precompute_service_features`: A speed optimization — computes service intensity (routes, trips, headways, service span) for *all* GTFS stops once and stores them in a dictionary.
- `get_service_features`: Fetches those precomputed values for existing stops, or infers nearby service for candidate points within 500m.

**6. Traffic / Congestion Features**
- `get_traffic_features`: Looks up daily traffic metrics (avg speed, congestion, trip count, variability) by matching each stop to its nearest traffic cell using a KD-tree.

**7. Derived / Composite Metrics**
- `get_derived_features`: Builds high-level engineered features such as demand-supply ratio, equity score, coverage efficiency, and network accessibility.

**8. Full Feature Extraction**
- `extract_stop_features`:  
  The main wrapper function that combines all the above.  
  Given a stop’s lat/lon (or candidate point), it computes **every feature**: road, spacing, population, traffic, GTFS service, ward context, derived metrics, and whether the ward is a benchmark.

**9. Candidate Stop Generation**
- `generate_ward_candidates`: Creates negative samples (candidate stops) inside each ward by randomly placing points away from existing stops, then computing the same feature set.  
  This is used for training ML models that require both positive (existing stops) and negative (candidate) examples.

Overall, these functions form the complete feature engineering pipeline for the stop-quality and stop-recommendation modeling workflow.


In [4]:
# helper functions
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def get_road_features(lat, lon, G, nodes_gdf):
    try:
        nearest_node = ox.distance.nearest_nodes(G, lon, lat)
        degree = G.degree(nearest_node)
        is_intersection = degree >= 3
       
        # get road type from edges
        edges = list(G.edges(nearest_node, data=True))
        if edges:
            road_type = edges[0][2].get('highway', 'unknown')
            if isinstance(road_type, list):
                road_type = road_type[0]
        else:
            road_type = 'unknown'
       
        # distance to major road
        major_edges = [(u, v, d) for u, v, d in G.edges(data=True)
                       if d.get('highway') in ['primary', 'secondary', 'trunk']]
        if major_edges:
            major_nodes = set([u for u, v, d in major_edges] + [v for u, v, d in major_edges])
            major_coords = nodes_gdf.loc[list(major_nodes), ['y', 'x']].values
            dists = cdist([[lat, lon]], major_coords)[0]
            dist_to_major = dists.min() * 111000 # to meters
        else:
            dist_to_major = np.nan
           
        return {
            'nearest_node_degree': degree,
            'is_intersection': is_intersection,
            'road_type': road_type,
            'distance_to_major_road': dist_to_major
        }
    except:
        return {
            'nearest_node_degree': np.nan,
            'is_intersection': False,
            'road_type': 'unknown',
            'distance_to_major_road': np.nan
        }

def get_stop_spacing_features(lat, lon, stop_tree, k=4):
    distances, indices = stop_tree.query([lat, lon], k=k)
   
    # if this is an existing stop, distances[0] = 0, so use [1:4]
    # if candidate, use [0:3]
    if distances[0] < 0.0001: # existing stop
        relevant_dists = distances[1:4] * 111000 # to meters
    else: # candidate
        relevant_dists = distances[0:3] * 111000
   
    return {
        'distance_to_nearest_stop': relevant_dists[0] if len(relevant_dists) > 0 else np.nan,
        'distance_to_2nd_nearest': relevant_dists[1] if len(relevant_dists) > 1 else np.nan,
        'distance_to_3rd_nearest': relevant_dists[2] if len(relevant_dists) > 2 else np.nan,
        'avg_spacing_3_nearest': np.mean(relevant_dists),
        'spacing_regularity': np.std(relevant_dists)
    }

def get_stops_in_radius(lat, lon, stop_tree, stop_coords, radius_km):
    indices = stop_tree.query_ball_point([lat, lon], radius_km / 111)
    return len(indices)

def get_population_features(lat, lon, ward, df1_agg):
    ward_data = df1_agg[df1_agg['ward'] == ward]
    if len(ward_data) == 0:
        return {
            'pop_within_500m': 0,
            'pop_within_1km': 0,
            'pop_density_500m': 0,
            'pop_not_served_nearby': 0,
            'poverty_rate_weighted_pop': 0
        }
   
    ward_data = ward_data.iloc[0]
    pop_density = ward_data['pop_density']
    poverty_rate = ward_data['poverty_rate']
   
    area_500m = np.pi * 0.5**2
    area_1km = np.pi * 1**2
   
    pop_500m = pop_density * area_500m
    pop_1km = pop_density * area_1km
   
    return {
        'pop_within_500m': int(pop_500m),
        'pop_within_1km': int(pop_1km),
        'pop_density_500m': pop_density,
        'pop_not_served_nearby': int(ward_data.get('pop_not_served', 0) * 0.3), # approximate
        'poverty_rate_weighted_pop': pop_500m * (poverty_rate / 100)
    }

# SPEEDUP: Precompute service features for all existing stops once (vectorized)
def precompute_service_features(gtfs_stop_times, gtfs_trips, PEAK_HOURS):
    print("Precomputing service features for all stops...")
    service_dict = {}
    stop_times = gtfs_stop_times.copy()
    stop_times['hour'] = pd.to_datetime(stop_times['arrival_time'], format='%H:%M:%S', errors='coerce').dt.hour
    stop_times = stop_times.dropna(subset=['hour'])
    
    for stop_id in tqdm(stop_times['stop_id'].unique(), desc="Precomputing stops"):
        st = stop_times[stop_times['stop_id'] == stop_id]
        if len(st) == 0:
            service_dict[stop_id] = {
                'route_count_serving': 0,
                'trips_per_day': 0,
                'trips_per_hour_peak': 0,
                'trips_per_hour_offpeak': 0,
                'avg_headway_minutes': np.nan,
                'service_span_hours': 0,
                'routes_within_500m': 0
            }
            continue
        
        trips_at_stop = st['trip_id'].unique()
        route_count = gtfs_trips[gtfs_trips['trip_id'].isin(trips_at_stop)]['route_id'].nunique()
        trips_per_day = len(st)
        
        peak_trips = st[st['hour'].isin(PEAK_HOURS)]
        offpeak_trips = st[~st['hour'].isin(PEAK_HOURS)]
        
        trips_per_hour_peak = len(peak_trips) / len(PEAK_HOURS) if len(PEAK_HOURS) > 0 else 0
        trips_per_hour_offpeak = len(offpeak_trips) / (24 - len(PEAK_HOURS)) if len(offpeak_trips) > 0 else 0
        
        avg_headway = (24 * 60) / trips_per_day if trips_per_day > 0 else np.nan
        service_span = st['hour'].max() - st['hour'].min() if len(st) > 0 else 0
        
        service_dict[stop_id] = {
            'route_count_serving': route_count,
            'trips_per_day': trips_per_day,
            'trips_per_hour_peak': trips_per_hour_peak,
            'trips_per_hour_offpeak': trips_per_hour_offpeak,
            'avg_headway_minutes': avg_headway,
            'service_span_hours': service_span,
            'routes_within_500m': route_count  # approximation
        }
    print("✓ Service features precomputed.")
    return service_dict

def get_service_features(stop_id, service_dict, gtfs_stop_times, gtfs_trips, all_stops_gdf, lat, lon):
    if pd.isna(stop_id) or stop_id not in service_dict:
        # candidate location - check nearby routes
        nearby = all_stops_gdf[
            np.sqrt((all_stops_gdf['stop_lat'] - lat)**2 +
                   (all_stops_gdf['stop_lon'] - lon)**2) < 0.005
        ]
        if len(nearby) > 0:
            nearby_stop_ids = nearby['stop_id'].values
            nearby_stop_times = gtfs_stop_times[gtfs_stop_times['stop_id'].isin(nearby_stop_ids)]
            nearby_trips = nearby_stop_times['trip_id'].unique()
            routes_within_500m = gtfs_trips[gtfs_trips['trip_id'].isin(nearby_trips)]['route_id'].nunique()
        else:
            routes_within_500m = 0
       
        return {
            'route_count_serving': 0,
            'trips_per_day': 0,
            'trips_per_hour_peak': 0,
            'trips_per_hour_offpeak': 0,
            'avg_headway_minutes': np.nan,
            'service_span_hours': 0,
            'routes_within_500m': routes_within_500m
        }
   
    return service_dict[stop_id]

# Update get_traffic_features to use daily lookup (simpler, no hourly filter)
def get_traffic_features(lat, lon, model2_daily, cell_daily_tree, cell_daily_ids):
    # Find nearest cell
    try:
        distances, indices = cell_daily_tree.query([lat, lon], k=1)
        # Handle scalar return for k=1 (scipy quirk)
        if np.isscalar(indices):
            nearest_cell_idx = int(indices)
        else:
            nearest_cell_idx = int(indices[0])
        nearest_cell = cell_daily_ids[nearest_cell_idx]
    except Exception as e:
        print(f"Query failed for ({lat}, {lon}): {e}")  # Optional: Remove after debug
        return {k: np.nan for k in [
            'avg_speed_daily', 'avg_speed_peak', 'avg_speed_offpeak',
            'congestion_pct_daily', 'congestion_pct_peak',
            'trip_count_daily', 'trip_count_peak',
            'demand_variability_cv', 'dominant_congestion_level'
        ]}
   
    # Direct lookup - no hourly agg needed
    cell_rows = model2_daily[model2_daily['cell_id'] == nearest_cell]
    if len(cell_rows) == 0:
        return {k: np.nan for k in [
            'avg_speed_daily', 'avg_speed_peak', 'avg_speed_offpeak',
            'congestion_pct_daily', 'congestion_pct_peak',
            'trip_count_daily', 'trip_count_peak',
            'demand_variability_cv', 'dominant_congestion_level'
        ]}
    
    cell_row = cell_rows.iloc[0]
   
    return {
        'avg_speed_daily': cell_row['avg_speed_daily'],
        'avg_speed_peak': cell_row['avg_speed_peak'],
        'avg_speed_offpeak': cell_row['avg_speed_offpeak'],
        'congestion_pct_daily': cell_row['congestion_pct_daily'],
        'congestion_pct_peak': cell_row['congestion_pct_peak'],
        'trip_count_daily': cell_row['trip_count_daily'],
        'trip_count_peak': cell_row['trip_count_peak'],
        'demand_variability_cv': cell_row['demand_variability_cv'],
        'dominant_congestion_level': cell_row['dominant_congestion_level']
    }

def get_ward_features(ward, ward_features_df):
    ward_data = ward_features_df[ward_features_df['ward'] == ward]
    if len(ward_data) == 0:
        return {
            'ward_pct_access': 0,
            'ward_population': 0,
            'ward_pop_density': 0,
            'ward_poverty_rate': 0,
            'ward_pop_not_served': 0,
            'is_benchmark_ward': False,
            'ward_service_per_capita': 0,
            'ward_category': 'unknown'
        }
   
    ward_data = ward_data.iloc[0]
    return {
        'ward_pct_access': ward_data.get('pct_access', 0),
        'ward_population': ward_data.get('population', 0),
        'ward_pop_density': ward_data.get('pop_density', 0),
        'ward_poverty_rate': ward_data.get('poverty_rate', 0),
        'ward_pop_not_served': ward_data.get('pop_not_served', 0),
        'is_benchmark_ward': ward_data.get('is_benchmark', False),
        'ward_service_per_capita': ward_data.get('service_per_capita', 0),
        'ward_category': ward_data.get('ward_category', 'unknown')
    }

def get_spatial_features(lat, lon):
    distance_to_cbd = haversine_km(lat, lon, CBD_COORDS[0][0], CBD_COORDS[0][1])
    return {
        'distance_to_cbd': distance_to_cbd
    }

def get_derived_features(features):
    stops_1km = features.get('stops_within_1km', 1)
    ward_access = features.get('ward_pct_access', 1)
    trips_per_day = features.get('trips_per_day', 0)
    pop_500m = features.get('pop_within_500m', 1)
    pop_not_served = features.get('pop_not_served_nearby', 0)
    poverty_weighted = features.get('poverty_rate_weighted_pop', 0)
    degree = features.get('nearest_node_degree', 1)
    dist_major = features.get('distance_to_major_road', 1)
   
    return {
        'coverage_efficiency_nearby': ward_access / (stops_1km + 1),
        'demand_supply_ratio': pop_500m / (trips_per_day + 1),
        'network_accessibility': degree / (dist_major + 1),
        'equity_score': pop_not_served * poverty_weighted
    }

def extract_stop_features(stop_data, G, nodes_gdf, stop_tree, stop_coords,
                          df1_agg, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops,
                          model2_daily, cell_daily_tree, cell_daily_ids, ward_features):  # Updated params
    idx, stop = stop_data
   
    lat, lon = stop['stop_lat'], stop['stop_lon']
    stop_id = stop['stop_id']
    ward = stop.get('ward', 'unknown')
   
    features = {
        'stop_id': stop_id,
        'lat': lat,
        'lon': lon,
        'ward': ward,
        'is_existing_stop': True
    }
   
    features.update(get_road_features(lat, lon, G, nodes_gdf))
    features.update(get_stop_spacing_features(lat, lon, stop_tree))
   
    stops_500m = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 0.5)
    stops_1km = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 1.0)
    features['stops_within_500m'] = stops_500m
    features['stops_within_1km'] = stops_1km
    features['stop_density_1km'] = stops_1km / (np.pi * 1**2)
   
    features.update(get_population_features(lat, lon, ward, df1_agg))
    features.update(get_service_features(stop_id, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops, lat, lon))  # SPEEDUP: Uses precomputed dict
    features.update(get_traffic_features(lat, lon, model2_daily, cell_daily_tree, cell_daily_ids))  # Updated
    features.update(get_ward_features(ward, ward_features))
    features.update(get_spatial_features(lat, lon))
    features.update(get_derived_features(features))
   
    features['is_good_stop'] = 1 if features['is_benchmark_ward'] else 0
   
    return features

# SPEEDUP: Function to generate candidates for a single ward (for parallel negatives)
def generate_ward_candidates(ward_data, n_negatives_per_ward, G, nodes_gdf, stop_tree, stop_coords,
                             df1_agg, gtfs_stop_times, gtfs_trips, gtfs_stops, service_dict,
                             model2_traffic, cell_tree, cell_ids, ward_features):
    ward, ward_stops = ward_data
    if len(ward_stops) == 0:
        return []
    
    lats = ward_stops['lat'].values
    lons = ward_stops['lon'].values
    
    lat_min, lat_max = lats.min() - 0.01, lats.max() + 0.01
    lon_min, lon_max = lons.min() - 0.01, lons.max() + 0.01
    
    candidates = []
    attempts = 0  # To avoid infinite loop if hard to place
    while len(candidates) < n_negatives_per_ward and attempts < n_negatives_per_ward * 10:
        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)
        
        # check not too close to existing stop
        dists, _ = stop_tree.query([lat, lon], k=1)
        dist_m = float(dists) * 111000 if isinstance(dists, (int, float)) else dists[0] * 111000
        
        if dist_m < 300:
            attempts += 1
            continue
        
        features = {
            'stop_id': f'CANDIDATE_{ward}_{len(candidates)}',
            'lat': lat,
            'lon': lon,
            'ward': ward,
            'is_existing_stop': False
        }
        
        features.update(get_road_features(lat, lon, G, nodes_gdf))
        features.update(get_stop_spacing_features(lat, lon, stop_tree))
        
        stops_500m = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 0.5)
        stops_1km = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 1.0)
        features['stops_within_500m'] = stops_500m
        features['stops_within_1km'] = stops_1km
        features['stop_density_1km'] = stops_1km / (np.pi * 1**2)
        
        features.update(get_population_features(lat, lon, ward, df1_agg))
        features.update(get_service_features(None, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops, lat, lon))
        features.update(get_traffic_features(lat, lon, model2_traffic, cell_tree, cell_ids))
        features.update(get_ward_features(ward, ward_features))
        features.update(get_spatial_features(lat, lon))
        features.update(get_derived_features(features))
        
        features['is_good_stop'] = 0
        
        candidates.append(features)
        attempts += 1
    
    return candidates

### v. Building Spatial Indices and Aggregates

Here we build fast spatial lookup structures to speed up all later feature computations.  
First, we create a KD-Tree of all GTFS stop coordinates, enabling quick nearest-stop and stop-density queries.  
We then build another KD-Tree for daily traffic cells so each stop or candidate location can be matched to its closest traffic cell in constant time.

Next, we aggregate ward-level population metrics using Polars (for speed), summarizing population, density, poverty rate, and underserved counts per ward.  
These pre-aggregated values are used by the population and ward feature functions.

Finally, we confirm that all spatial indices and aggregates are ready for use.

In [5]:
# build spatial indices
print("Building spatial indices...")
stop_coords = gtfs_stops[['stop_lat', 'stop_lon']].values
stop_tree = cKDTree(stop_coords)

# KDTree for traffic cells (daily version: lat first for query)
cell_daily_coords = model2_daily[['lat', 'lon']].values
cell_daily_tree = cKDTree(cell_daily_coords)
cell_daily_ids = model2_daily['cell_id'].values

# SPEEDUP: Use Polars for ward aggs
df1_pl = pl.from_pandas(df1.drop(columns=df1.select_dtypes(include=['geometry'])))  # Drop geo for agg
df1_agg_pl = df1_pl.group_by('ward').agg([
    pl.col('population').first(),
    pl.col('pop_density').first(),
    pl.col('poverty_rate').first(),
    pl.col('pop_not_served').first()
]).to_pandas()  # Back to pandas for compatibility
df1_agg = df1_agg_pl
print("✓ Spatial indices built.")

Building spatial indices...
✓ Spatial indices built.


### vi. Preparing Ward-Level Feature Table

Here we use Polars to aggregate all ward-level indicators needed for modeling.  
For each ward, we extract or summarize population, density, poverty rate, service coverage, Gini inequality, and transit activity (mean, std, and total trips per hour).  
We also compute service-per-capita metrics and preserve the overall service ranking.

After converting back to pandas, we add two important derived attributes:
- **is_benchmark**: marks wards with ≥70% access as high-performing reference wards  
- **ward_category**: classifies each ward into served/underserved categories based on access levels  

This table provides the contextual socio-economic and service-quality features used throughout feature engineering.

In [6]:
# prepare ward features (Polars)
print("Preparing ward features...")
ward_features_pl = df1_pl.group_by('ward').agg([
    pl.col('population').first().alias('population'),
    pl.col('pop_density').first().alias('pop_density'),
    pl.col('pct_access').first().alias('pct_access'),
    pl.col('pop_served').first().alias('pop_served'),
    pl.col('pop_not_served').first().alias('pop_not_served'),
    pl.col('coverage_ratio').first().alias('coverage_ratio'),
    pl.col('poverty_rate').first().alias('poverty_rate'),
    pl.col('subcounty_gini').first().alias('subcounty_gini'),
    pl.col('trips_per_hour').mean().alias('trips_per_hour_mean'),
    pl.col('trips_per_hour').std().alias('trips_per_hour_std'),
    pl.col('trips_per_hour').sum().alias('trips_per_hour_sum'),
    pl.col('trips_per_1k_pop_per_hour').mean().alias('service_per_capita'),
    pl.col('trips_per_1k_pop_per_hour').std().alias('trips_per_1k_pop_per_hour_std'),
    pl.col('service_rank_overall').first().alias('service_rank_overall')
])
ward_features = ward_features_pl.to_pandas()
ward_features['is_benchmark'] = ward_features['pct_access'] >= 70
ward_features['ward_category'] = pd.cut(
    ward_features['pct_access'],
    bins=[0, 50, 70, 90, 100],
    labels=['severely_underserved', 'underserved', 'adequately_served', 'well_served']
)
print("✓ Ward features prepared.")

Preparing ward features...
✓ Ward features prepared.


### vii. Testing Mode Configuration

Here we set up a lightweight testing mode so the full pipeline can run quickly during development.  
`TEST_MODE` limits the number of GTFS stops processed and reduces the number of negative (candidate) points generated per ward.  
This makes it easy to debug the feature extraction workflow without running the full, expensive computation.


In [7]:
# Testing mode
TEST_MODE = True  # Set to False for full run
N_TEST_STOPS = 100
N_TEST_NEG_PER_WARD = 10  # Small for speed; adjust as needed

### viii. Precomputing GTFS Service Features

Here we speed up the pipeline by precomputing service metrics (routes, trips, peak/off-peak frequency, headways, service span) for all stops.  
In testing mode, we only compute these features for a small subset of stop IDs to keep the process fast.  
The results are stored in `service_dict`, allowing instant lookup later during feature extraction.


In [8]:
# SPEEDUP: Precompute service (full or test subset)
print("Precomputing service features...")
if TEST_MODE:
    # Test mode: Precompute only on unique stop_ids from test stops (faster)
    test_stop_ids = gtfs_stops['stop_id'].unique()[:N_TEST_STOPS]  # Subset
    service_dict = precompute_service_features(gtfs_stop_times[gtfs_stop_times['stop_id'].isin(test_stop_ids)], gtfs_trips, PEAK_HOURS)
else:
    service_dict = precompute_service_features(gtfs_stop_times, gtfs_trips, PEAK_HOURS)
print("✓ Service features precomputed.")

Precomputing service features...
Precomputing service features for all stops...


Precomputing stops:   0%|          | 0/89 [00:00<?, ?it/s]

Precomputing stops: 100%|██████████| 89/89 [00:00<00:00, 195.02it/s]

✓ Service features precomputed.
✓ Service features precomputed.


> The service-feature precomputation completed successfully.  
In test mode, only 89 unique stops were processed, which finished quickly thanks to vectorized filtering and the progress-bar–driven loop.  
All service metrics;route counts, headways, peak/off-peak frequency, and service span; are now cached in `service_dict` for fast lookup during feature extraction.


### ix. Extracting Features for Existing GTFS Stops (Parallelized)

Here we generate the full feature set for all existing GTFS stops.  
We first prepare a list of stop records, and in test mode restrict this to a smaller subset for speed.  
We then use `functools.partial` to bind all shared data inputs to `extract_stop_features`, leaving only the stop row itself to be processed.

The work is distributed across multiple CPU cores using a process pool, allowing each stop’s features to be computed in parallel.  
As the pool runs, a `tqdm` progress bar shows real-time status.  
Finally, we collect the resulting feature dictionaries into a pandas DataFrame containing one row per stop.


In [9]:
# extract features for existing stops (parallelized)
print("Extracting features for existing stops...")
# prepare data
stop_data_list = list(gtfs_stops.iterrows())
if TEST_MODE:
    stop_data_list = stop_data_list[:N_TEST_STOPS]  # Subset to first N_TEST_STOPS
    print(f"TEST MODE: Using only {len(stop_data_list)} stops")
extract_func = partial(
    extract_stop_features,
    G=G,
    nodes_gdf=nodes_gdf,
    stop_tree=stop_tree,
    stop_coords=stop_coords,
    df1_agg=df1_agg,
    service_dict=service_dict,
    gtfs_stop_times=gtfs_stop_times,
    gtfs_trips=gtfs_trips,
    gtfs_stops=gtfs_stops,
    model2_daily=model2_daily,
    cell_daily_tree=cell_daily_tree,
    cell_daily_ids=cell_daily_ids,
    ward_features=ward_features
)
n_cores = cpu_count() - 1
print(f"Using {n_cores} cores for {len(stop_data_list)} stops...")
with Pool(n_cores) as pool:
    all_features = list(tqdm(
        pool.imap(extract_func, stop_data_list, chunksize=20),
        total=len(stop_data_list),
        desc="Processing stops"
    ))
stops_df = pd.DataFrame(all_features)
print(f"Processed {len(stops_df)} stops")

Extracting features for existing stops...
TEST MODE: Using only 100 stops
Using 7 cores for 100 stops...


Processing stops: 100%|██████████| 100/100 [00:59<00:00,  1.68it/s]


Processed 100 stops


> Feature extraction for existing stops completed.  
In test mode, 100 stops were processed using 7 CPU cores, with the parallel pipeline running at roughly 1.7 stops per second.  
All spatial, service, traffic, population, and ward-level features for these stops are now compiled into the `stops_df` dataframe.


### x. Merging Updated Population Buffer Data

Here we load the refined population-buffer dataset and merge it into the stop-level features.  
We first remove any older population columns to avoid conflicts, then join the new population metrics by `stop_id`.  
This ensures each stop has the updated 200m, 500m, and 1km population estimates for more accurate demand modeling.


In [10]:
# load and merge additional data
pop_df = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/stop_population_buffers.csv')
# drop old pop columns first
stops_df = stops_df.drop(columns=['pop_within_200m', 'pop_within_500m', 'pop_within_1km'], errors='ignore')
# merge new population data
stops_df = stops_df.merge(pop_df, on='stop_id', how='left')

### xi. Recomputing Derived Features Using Updated Population Data

Here we update the derived (engineered) metrics now that the refined population buffers have been merged.  
All calculations are done vectorized for speed-no `.apply()` calls.

We recompute coverage efficiency, demand–supply ratio, network accessibility, and the equity score using the newest population, service, and network fields.  
This ensures the downstream model uses the most accurate and consistent feature values.


In [11]:
# SPEEDUP: Vectorized recompute derived (no apply)
print("Recomputing derived features with real pop data...")
# Vector ops for derived (using merged pop columns directly)
stops_df['coverage_efficiency_nearby'] = stops_df['ward_pct_access'] / (stops_df['stops_within_1km'] + 1)
stops_df['demand_supply_ratio'] = stops_df['pop_within_500m'] / (stops_df['trips_per_day'] + 1)
stops_df['network_accessibility'] = stops_df['nearest_node_degree'] / (stops_df['distance_to_major_road'] + 1)
stops_df['equity_score'] = stops_df['pop_not_served_nearby'] * stops_df['poverty_rate_weighted_pop']
print("✓ Derived features recomputed with real pop data")
stops_df.head(2)

Recomputing derived features with real pop data...
✓ Derived features recomputed with real pop data


,stop_id,lat,lon,ward,is_existing_stop,nearest_node_degree,is_intersection,road_type,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,dominant_congestion_level,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,is_benchmark_ward,ward_service_per_capita,ward_category,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop,pop_within_200m,pop_within_500m,pop_within_1km
0,0001RLW,-1.290884,36.828242,Nairobi Central Ward,True,4,True,unclassified,185.404122,173.684507,231.853573,412.893917,272.810666,101.860683,9,57,18.143664,6034.16,0,947.843636,7,14,1.75,0,102.857143,1,7,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,1.324902,1.724138,372.666667,0.021459,0.0,1,799,5590,27759
1,0002KOJ,-1.281230,36.822596,Nairobi Central Ward,True,3,True,residential,20.568573,5.177885,21.286282,23.097604,16.520590,8.054520,14,66,21.008452,6034.16,0,947.843636,0,0,0.00,0,NaN,0,46,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.830851,1.492537,5161.000000,0.139091,0.0,1,757,5161,23044


### xii. Negative Example Generator (Candidate Stop Creation)

This function creates artificial “negative” examples locations where a stop *does not* currently exist but could theoretically be placed.  
These are used to train models that distinguish good stop locations from bad ones.

For each ward, we:
- Sample random points within the spatial bounds of that ward’s existing stops.  
- Reject points that fall too close to an existing stop (within 300m).  
- For each valid point, compute the full feature set: road connectivity, spacing to nearby stops, population buffers, traffic context, service proximity, ward attributes, and derived metrics.  
- Label each candidate as `is_good_stop = 0`.

This produces realistic, spatially grounded negative samples to complement the positive (existing) GTFS stops in model training.

In [12]:
# generate negative examples (parallelized)
# SPEEDUP: Function to generate candidates for a single ward (for parallel negatives) - FULL DEFINITION
def generate_ward_candidates(ward_data, n_negatives_per_ward, G, nodes_gdf, stop_tree, stop_coords,
                             df1_agg, gtfs_stop_times, gtfs_trips, gtfs_stops, service_dict,
                             model2_daily, cell_daily_tree, cell_daily_ids, ward_features):
    ward, ward_stops = ward_data
    if len(ward_stops) == 0:
        return []
    
    lats = ward_stops['lat'].values
    lons = ward_stops['lon'].values
    
    lat_min, lat_max = lats.min() - 0.01, lats.max() + 0.01
    lon_min, lon_max = lons.min() - 0.01, lons.max() + 0.01
    
    candidates = []
    attempts = 0  # To avoid infinite loop if hard to place
    while len(candidates) < n_negatives_per_ward and attempts < n_negatives_per_ward * 10:
        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)
        
        # check not too close to existing stop
        dists, _ = stop_tree.query([lat, lon], k=1)
        dist_m = float(dists) * 111000 if isinstance(dists, (int, float)) else dists[0] * 111000
        
        if dist_m < 300:
            attempts += 1
            continue
        
        features = {
            'stop_id': f'CANDIDATE_{ward}_{len(candidates)}',
            'lat': lat,
            'lon': lon,
            'ward': ward,
            'is_existing_stop': False
        }
        
        features.update(get_road_features(lat, lon, G, nodes_gdf))
        features.update(get_stop_spacing_features(lat, lon, stop_tree))
        
        stops_500m = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 0.5)
        stops_1km = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 1.0)
        features['stops_within_500m'] = stops_500m
        features['stops_within_1km'] = stops_1km
        features['stop_density_1km'] = stops_1km / (np.pi * 1**2)
        
        features.update(get_population_features(lat, lon, ward, df1_agg))
        features.update(get_service_features(None, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops, lat, lon))
        features.update(get_traffic_features(lat, lon, model2_daily, cell_daily_tree, cell_daily_ids))
        features.update(get_ward_features(ward, ward_features))
        features.update(get_spatial_features(lat, lon))
        features.update(get_derived_features(features))
        
        features['is_good_stop'] = 0
        
        candidates.append(features)
        attempts += 1
    
    return candidates

### xiii. Generating Negative Examples (Parallelized)

Here we create the negative (non-stop) samples for each ward.  
In test mode, we only generate candidates for the wards present in the limited `stops_df` subset; otherwise, we cover all wards with existing stops.

We bind all necessary data to the `generate_ward_candidates` function using `partial`, then run it in parallel-one ward per task.  
Each ward receives a fixed number of candidate points (`n_negatives_per_ward`), and we limit the number of cores so we don’t over-parallelize when only a few wards are involved.

After parallel execution, we flatten the list of ward-level results into a single DataFrame of negative examples.  
This dataset provides the contrasting examples needed for training a binary classifier on stop suitability.


In [13]:
print("Generating negative examples...")
n_negatives_per_ward = N_TEST_NEG_PER_WARD if TEST_MODE else 50  # Reduce for test
if TEST_MODE:
    # Only wards from the test stops_df
    test_wards = stops_df['ward'].unique()
    ward_data_list = [(ward, stops_df[stops_df['ward'] == ward]) for ward in test_wards]
    print(f"TEST MODE: Generating negatives only for {len(ward_data_list)} wards from test stops")
else:
    ward_data_list = [(ward, stops_df[stops_df['ward'] == ward]) for ward in ward_features['ward'].unique() if len(stops_df[stops_df['ward'] == ward]) > 0]
gen_func = partial(
    generate_ward_candidates,
    n_negatives_per_ward=n_negatives_per_ward,
    G=G,
    nodes_gdf=nodes_gdf,
    stop_tree=stop_tree,
    stop_coords=stop_coords,
    df1_agg=df1_agg,
    gtfs_stop_times=gtfs_stop_times,
    gtfs_trips=gtfs_trips,
    gtfs_stops=gtfs_stops,
    service_dict=service_dict,
    model2_daily=model2_daily,
    cell_daily_tree=cell_daily_tree,
    cell_daily_ids=cell_daily_ids,
    ward_features=ward_features
)
n_cores_neg = min(n_cores, len(ward_data_list))  # Don't over-parallelize if few wards
print(f"Parallel generating {n_negatives_per_ward} negatives per ward across {len(ward_data_list)} wards...")
with Pool(n_cores_neg) as pool:
    ward_candidates = list(tqdm(
        pool.imap(gen_func, ward_data_list, chunksize=1),  # One ward per task
        total=len(ward_data_list),
        desc="Generating negatives"
    ))

negatives = [cand for ward_cands in ward_candidates for cand in ward_cands]
negatives_df = pd.DataFrame(negatives)
print(f"Generated {len(negatives_df)} negative examples")
negatives_df.head(2)

Generating negative examples...
TEST MODE: Generating negatives only for 10 wards from test stops
Parallel generating 10 negatives per ward across 10 wards...


Generating negatives: 100%|██████████| 10/10 [01:21<00:00,  8.19s/it]


Generated 100 negative examples


,stop_id,lat,lon,ward,is_existing_stop,nearest_node_degree,is_intersection,road_type,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_within_500m,pop_within_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,dominant_congestion_level,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,is_benchmark_ward,ward_service_per_capita,ward_category,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop
0,CANDIDATE_Nairobi Central Ward_0,-1.289001,36.821147,Nairobi Central Ward,False,4,True,secondary_link,271.728297,340.736684,358.883427,369.795063,356.471725,11.984979,11,48,15.278875,4739,18956,6034.16,0,947.843636,0,0,0,0,NaN,0,35,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.525528,2.040816,4739.0,0.014667,0.0,0
1,CANDIDATE_Nairobi Central Ward_1,-1.298724,36.841741,Nairobi Central Ward,False,6,True,residential,354.408282,387.360546,436.383778,511.165125,444.969817,50.906342,2,30,9.549297,4739,18956,6034.16,0,947.843636,0,0,0,0,NaN,0,4,17.243301,11.884734,20.101203,63.471160,93.750000,190,84.0,0.583137,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,3.052997,3.225806,4739.0,0.016882,0.0,0


>Negative example generation completed.  
In test mode, only 10 wards were included, with 10 candidate points generated per ward.  
The parallel process finished in about 80 seconds, producing a total of 100 high-quality negative samples for model training.


### xiv. Combining and Saving the Training Dataset

Here we merge the positive samples (existing stops) with the negative samples (candidate locations) into one final training dataset.  
The output is saved to a test file or full-production file depending on the mode.  
A quick summary is printed showing total samples, the balance of positive vs. negative labels, and the save path.  
This completes the data preparation pipeline for stop-quality modeling.


In [14]:
# combine and save
print("Combining datasets...")
final_df = pd.concat([stops_df, negatives_df], ignore_index=True)
# save
filename = '/home/dataopske/Desktop/jav/data/training_data/stop_features_test.csv' if TEST_MODE else 'stop_features_complete.csv'
final_df.to_csv(filename, index=False)
print(f"\n Complete! ({'TEST MODE' if TEST_MODE else 'FULL RUN'})")
print(f"Total samples: {len(final_df)}")
print(f" Existing stops: {len(stops_df)}")
print(f" Candidates: {len(negatives_df)}")
print(f" Positive labels: {final_df['is_good_stop'].sum()}")
print(f" Negative labels: {(final_df['is_good_stop'] == 0).sum()}")
print(f"\nSaved to: {filename}")
final_df.head()

Combining datasets...

 Complete! (TEST MODE)
Total samples: 200
 Existing stops: 100
 Candidates: 100
 Positive labels: 20
 Negative labels: 180

Saved to: /home/dataopske/Desktop/jav/data/training_data/stop_features_test.csv


,stop_id,lat,lon,ward,is_existing_stop,nearest_node_degree,is_intersection,road_type,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,dominant_congestion_level,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,is_benchmark_ward,ward_service_per_capita,ward_category,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop,pop_within_200m,pop_within_500m,pop_within_1km
0,0001RLW,-1.290884,36.828242,Nairobi Central Ward,True,4,True,unclassified,185.404122,173.684507,231.853573,412.893917,272.810666,101.860683,9,57,18.143664,6034.16,0,947.843636,7,14,1.75,0,102.857143,1,7,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,1.324902,1.724138,372.666667,0.021459,0.0,1,799.0,5590,27759
1,0002KOJ,-1.281230,36.822596,Nairobi Central Ward,True,3,True,residential,20.568573,5.177885,21.286282,23.097604,16.520590,8.054520,14,66,21.008452,6034.16,0,947.843636,0,0,0.00,0,NaN,0,46,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.830851,1.492537,5161.000000,0.139091,0.0,1,757.0,5161,23044
2,0003NGR,-1.274395,36.823806,Ngara Ward Ward,True,3,True,primary,58.275689,27.998313,31.421882,53.807529,37.742574,11.445298,23,64,20.371833,9887.15,0,1553.069890,0,0,0.00,0,NaN,0,26,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,25589.814342,9887.15,20.0,0.0,True,62.733293,well_served,1.523563,1.538462,7734.000000,0.050611,0.0,1,1240.0,7734,26486
3,0004ODN,-1.282769,36.825032,Nairobi Central Ward,True,3,True,tertiary,67.183239,14.009785,64.101044,81.767482,53.292770,28.698321,26,76,24.191551,6034.16,0,947.843636,0,0,0.00,0,NaN,0,54,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.959720,1.298701,4809.000000,0.043999,0.0,1,757.0,4809,25851
4,0005AMB,-1.285963,36.826048,Nairobi Central Ward,True,4,True,secondary_link,60.725155,17.426995,68.153819,103.673253,63.084689,35.391866,29,65,20.690143,6034.16,0,947.843636,0,0,0.00,0,NaN,0,65,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.984804,1.515152,4731.000000,0.064803,0.0,1,757.0,4731,26761


**Summary**

This notebook constructs a complete stop-level dataset by combining GTFS data, OSM road networks, traffic analytics, ward-level socio-economic indicators, and spatial population models.  
Both existing GTFS stops and synthetic “negative” candidate stops are analyzed to produce a balanced dataset for binary classification or GNN node classification.

**Main Steps**

**1. Load Core Inputs**
- GTFS feed (routes, trips, stop times, shapes, stops)  
- Ward-level socio-economic and service metrics  
- Processed daily traffic grid  
- Updated population buffer metrics  
- OSM road network for Nairobi  

**2. Build Spatial Index Structures**
- KD-tree for GTFS stops → fast nearest-stop and spacing queries  
- KD-tree for traffic cells → instant congestion lookup  
- Polars-based ward aggregations → high-speed population summarization  

**3. Prepare Ward-Level Feature Table**
- Population, density, access %, poverty rate, Gini inequality, service-per-capita  
- Benchmark ward flag and ward category (underserved → well-served)

**4. Precompute Transit Service Features**
Stop-level GTFS service metrics:
- Route count  
- Trips per day  
- Peak/off-peak frequency  
- Average headway  
- Service span  
This is done upfront for speed.

**5. Extract Features for Existing GTFS Stops (Parallelized)**
For every stop, compute:
- Road connectivity (intersection degree, road type, major-road proximity)  
- Stop spacing & density (nearest-stop distances, local density)  
- Population features (buffers, density, underserved population)  
- Traffic features (avg speed, congestion %, trip counts)  
- Ward features (poverty, access %, categories)  
- Distance to CBD  
- Derived features (supply/demand ratio, equity score, network access)

**6. Generate Negative Candidate Examples (Parallel)**
For each ward:
- Randomly propose candidate points  
- Reject those within 300m of existing stops  
- Compute the same feature set as real stops  
- Label as `is_good_stop = 0`  
These negatives enable training of supervised models.

**7. Merge, Recompute Derived Metrics, and Save**
- Combine positives + negatives  
- Incorporate updated population buffers  
- Recompute derived metrics using real population  
- Save to CSV (test mode or full mode)

This produces a clean, balanced training dataset ready for:
- Random Forest / XGBoost stop-quality classifiers  
- Semi-supervised or supervised GNNs  
- Stop placement optimization algorithms  
- Accessibility analysis and policy scenario modeling  